# Configuration Guide

This notebook explains metaeval's configuration system and all available settings.

In [ ]:
from metaeval.core.config import (
    Config,
    JudgeConfig,
    StatsConfig,
    OllamaConfig,
    PathsConfig,
    get_config,
    set_config,
    reset_config,
    get_config_path,
    generate_default_config,
)

## Configuration Overview

Metaeval uses a hierarchical configuration system with these main sections:

1. **JudgeConfig**: LLM-as-a-Judge settings
2. **StatsConfig**: Statistical analysis settings
3. **OllamaConfig**: Local model settings
4. **PathsConfig**: File path settings

## Getting Current Configuration

In [ ]:
# Get the current configuration
config = get_config()

# View as dictionary
config_dict = config.to_dict()
print("Current Configuration:")
for section, values in config_dict.items():
    print(f"\n[{section}]")
    if isinstance(values, dict):
        for key, value in values.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {values}")

## Judge Configuration

In [ ]:
# Default judge settings
judge_config = JudgeConfig()

print("Judge Configuration:")
print(f"  temperature: {judge_config.temperature}")
print(f"    - Controls randomness in LLM output")
print(f"    - 0.0 = deterministic, higher = more creative")
print(f"")
print(f"  max_tokens: {judge_config.max_tokens}")
print(f"    - Maximum tokens to generate in response")
print(f"")
print(f"  timeout: {judge_config.timeout}")
print(f"    - Request timeout in seconds")
print(f"")
print(f"  default_prompt: {judge_config.default_prompt}")
print(f"    - Default prompt style for judging")
print(f"")
print(f"  default_provider: {judge_config.default_provider}")
print(f"    - Default LLM provider (ollama, openai, anthropic, openrouter)")

## Stats Configuration

In [ ]:
# Default stats settings
stats_config = StatsConfig()

print("Statistical Analysis Configuration:")
print(f"  alpha: {stats_config.alpha}")
print(f"    - Significance level for hypothesis tests")
print(f"    - Lower = more strict (e.g., 0.01)")
print(f"")
print(f"  bootstrap_iterations: {stats_config.bootstrap_iterations}")
print(f"    - Number of bootstrap resamples for confidence intervals")
print(f"")
print(f"  confidence_level: {stats_config.confidence_level}")
print(f"    - Confidence level for intervals (0.95 = 95%)")
print(f"")
print(f"  random_seed: {stats_config.random_seed}")
print(f"    - Random seed for reproducibility")
print(f"")
print(f"  effect_size_thresholds:")
for level, value in stats_config.effect_size_thresholds.items():
    print(f"    - {level}: {value}")

## Ollama Configuration

In [ ]:
# Default Ollama settings
ollama_config = OllamaConfig()

print("Ollama Configuration:")
print(f"  host: {ollama_config.host}")
print(f"    - Ollama server hostname")
print(f"")
print(f"  port: {ollama_config.port}")
print(f"    - Ollama server port")
print(f"")
print(f"  default_model: {ollama_config.default_model}")
print(f"    - Default model for judging")
print(f"")
print(f"  auto_pull: {ollama_config.auto_pull}")
print(f"    - Automatically pull missing models")

## Customizing Configuration

In [ ]:
# Create custom configuration
custom_config = Config()

# Modify judge settings
custom_config.judge.temperature = 0.1
custom_config.judge.max_tokens = 4096
custom_config.judge.default_provider = 'openai'

# Modify stats settings
custom_config.stats.alpha = 0.01  # More strict
custom_config.stats.bootstrap_iterations = 5000  # Faster

# Modify Ollama settings
custom_config.ollama.host = 'gpu-server.local'
custom_config.ollama.port = 11434

# Set as global config
set_config(custom_config)
print("Custom configuration applied!")

In [ ]:
# Verify changes
current = get_config()
print(f"Judge temperature: {current.judge.temperature}")
print(f"Stats alpha: {current.stats.alpha}")
print(f"Ollama host: {current.ollama.host}")

In [ ]:
# Reset to defaults
reset_config()
default = get_config()
print(f"Reset to defaults:")
print(f"  Judge temperature: {default.judge.temperature}")
print(f"  Stats alpha: {default.stats.alpha}")

## Configuration File

Create a persistent config file at `~/.metaeval/config.yaml`:

In [ ]:
# Get config file path
config_path = get_config_path()
print(f"Config file location: {config_path}")

In [ ]:
# Generate default config content
default_config = generate_default_config()
print("Default config.yaml:")
print("="*60)
print(default_config)

## Loading from YAML

In [ ]:
import tempfile
import yaml
from pathlib import Path

# Create a temporary config file
temp_config = {
    'judge': {
        'temperature': 0.2,
        'max_tokens': 3000,
    },
    'stats': {
        'alpha': 0.05,
        'bootstrap_iterations': 1000,
    },
}

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    yaml.dump(temp_config, f)
    temp_path = f.name

# Load from YAML
loaded_config = Config.from_yaml(temp_path)
print(f"Loaded from YAML:")
print(f"  Judge temperature: {loaded_config.judge.temperature}")
print(f"  Judge max_tokens: {loaded_config.judge.max_tokens}")
print(f"  Stats alpha: {loaded_config.stats.alpha}")

# Cleanup
Path(temp_path).unlink()

## Environment Variables

Configuration can also be set via environment variables:

```bash
# API Keys
export OPENAI_API_KEY=sk-...
export ANTHROPIC_API_KEY=sk-ant-...
export OPENROUTER_API_KEY=sk-or-...

# Ollama settings
export OLLAMA_HOST=localhost
export OLLAMA_PORT=11434

# Stats settings
export METAEVAL_ALPHA=0.01
export METAEVAL_BOOTSTRAP_ITERATIONS=5000
```

## CLI Configuration

```bash
# Show current configuration
metaeval config show

# Show config file path
metaeval config path

# Create default config file
metaeval config init

# Create config at custom location
metaeval config init --output ./custom_config.yaml
```

## Best Practices

1. **For reproducibility**: Set `random_seed` to a fixed value
2. **For faster testing**: Reduce `bootstrap_iterations` during development
3. **For production**: Use `temperature=0.0` for deterministic judging
4. **For strict analysis**: Lower `alpha` to 0.01 or 0.001
5. **Store API keys** in environment variables, not config files